In [1]:
pip install networkx pyvis

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 756.0/756.0 kB 20.2 MB/s eta 0:00:00
Note: you may need to restart the kernel to use updated packages.


In [2]:
import pandas as pd
import networkx as nx
from pyvis.network import Network
import logging
import os

In [3]:
class TrainingConfig:
    POS_DATA_FILE = "/kaggle/input/lrec-tcs-attack-support/sentencePair.txt"
    NEG_DATA_FILE = "/kaggle/input/lrec-tcs-attack-support/sentencePair_neg.txt"
    DEVICE = "cpu"

def parse_lrec_line(line: str):
    parts = line.strip().split("\t")
    fname_indices = [i for i, p in enumerate(parts) if p.endswith(".txt")]
    if len(fname_indices) != 2: return None
    try:
        sentpair_id = int(parts[0])
        sent1 = " ".join(parts[fname_indices[0] + 2 : fname_indices[1]]).strip('"')
        sent2 = " ".join(parts[fname_indices[1] + 2 : len(parts) - 2]).strip('"')
        label = parts[-1]
    except (ValueError, IndexError): return None
    return {"id": sentpair_id, "sent1": sent1, "sent2": sent2, "label": label}

# to load the full dataset for clustering
def load_full_data(config: TrainingConfig):
    logging.info("Loading full dataset for clustering analysis...")
    rows = []
    for filepath in [config.POS_DATA_FILE, config.NEG_DATA_FILE]:
        if not os.path.exists(filepath):
            logging.error(f"Data file not found at '{filepath}'.")
            return None
        with open(filepath, "r", encoding="utf-8") as f:
            for line in f:
                if parsed := parse_lrec_line(line): rows.append(parsed)
    df = pd.DataFrame(rows)
    label_map = {"SUPPORT": 0, "ATTACK": 1, "NO_REL": 2}
    df["label_id"] = df["label"].map(label_map)
    df = df.dropna(subset=['id', 'sent1', 'sent2', 'label_id', 'label'])
    logging.info(f"Full dataset loaded. Total samples: {len(df)}")
    return df.reset_index(drop=True)

In [4]:
logger = logging.getLogger(__name__)

def normalize_sentence_text(text: str) -> str:
    """
    Cleans and standardizes sentence text for consistent node matching.
    """
    if not isinstance(text, str):
        return ""
    
    text = text.lower()  # Convert to lowercase
    text = text.strip()    # Remove leading/trailing whitespace
    text = re.sub(r'\s+', ' ', text) # Replace multiple spaces with a single space
    
    return text

def perform_graph_analysis(df: pd.DataFrame, file_name="argument_map.html", text_normalize_flag = False):
    """
    Creates and visualizes a graph of sentence relationships.
    Nodes are sentences, edges are ATTACK/SUPPORT relations.
    """
    logger.info("Starting Graph-based Clustering Analysis...")

    # Filter for only ATTACK and SUPPORT relationships as requested
    graph_df = df[df['label'].isin(['ATTACK', 'SUPPORT'])].copy()
    
    if len(graph_df) == 0:
        logger.warning("No ATTACK or SUPPORT relations found to build the graph.")
        return

    # For visualization clarity, let's work with a sample of the graph
    # Building a graph of 40k nodes can be slow and hard to visualize
    if len(graph_df) > 2000:
        logger.info(f"Full graph has {len(graph_df)} edges. Visualizing a sample of 2000 for clarity.")
        graph_df = graph_df.sample(n=2000, random_state=42)

    G = nx.Graph()

    # Add edges with attributes
    for _, row in graph_df.iterrows():
        # Using the unique sentence ID or the text itself as a node identifier.
        # Text is more readable in the graph.
        if text_normalize_flag:
            sent1_node = normalize_sentence_text(row['sent1'])
            sent2_node = normalize_sentence_text(row['sent2'])
        else:
            sent1_node = row['sent1']
            sent2_node = row['sent2']
        relation = row['label']
        
        if not sent1_node or not sent2_node:
            continue

        G.add_edge(sent1_node, sent2_node, title=relation, label=relation)

    logger.info(f"Graph created with {G.number_of_nodes()} nodes and {G.number_of_edges()} edges.")

    # --- Interactive Visualization with Pyvis ---
    net = Network(notebook=True, height="750px", width="100%", bgcolor="#222222", font_color="white", cdn_resources='in_line')
    
    # Load the NetworkX graph into Pyvis
    net.from_nx(G)

    # Customize edge colors
    for edge in net.edges:
        if edge['label'] == 'ATTACK':
            edge['color'] = '#FF5733' # Red
        elif edge['label'] == 'SUPPORT':
            edge['color'] = '#33FF57' # Green

    # Add physics layout options for better interaction
    net.show_buttons(filter_=['physics'])
    
    # Save and show the interactive HTML file
    try:
        net.save_graph(file_name)
        logger.info(f"Interactive graph saved to '{file_name}'. Open this file in your browser to explore.")
    except Exception as e:
        logger.error(f"Could not save the graph: {e}")

    # Save the full graph structure to a GraphML file for use in Gephi
    try:
        graphml_file = "full_argument_graph.graphml"
        nx.write_graphml(G, graphml_file)
        logger.info(f"Full graph data saved to '{graphml_file}'")
    except Exception as e:
        logger.error(f"Could not save GraphML file: {e}")



In [5]:
if __name__ == '__main__':
    config = TrainingConfig()
    # Setup logger
    logger = logging.getLogger(__name__)
    logger.setLevel(logging.INFO)
    formatter = logging.Formatter('%(asctime)s - %(levelname)s - %(message)s')
    if not logger.handlers:
        file_handler = logging.FileHandler("data_analysis3.log")
        stream_handler = logging.StreamHandler()
        file_handler.setFormatter(formatter)
        stream_handler.setFormatter(formatter)
        logger.addHandler(file_handler)
        logger.addHandler(stream_handler)

    logger.info(f"Using device: {config.DEVICE}")

    # ---  ANALYSIS SECTION ---
    
    full_df = load_full_data(config)
    if full_df is not None:
        perform_graph_analysis(full_df)

2025-12-15 12:11:57,523 - INFO - Using device: cpu
2025-12-15 12:11:58,112 - INFO - Starting Graph-based Clustering Analysis...
2025-12-15 12:11:58,126 - INFO - Full graph has 20506 edges. Visualizing a sample of 2000 for clarity.
2025-12-15 12:11:58,250 - INFO - Graph created with 3940 nodes and 1991 edges.
2025-12-15 12:11:58,907 - INFO - Interactive graph saved to 'argument_map.html'. Open this file in your browser to explore.
2025-12-15 12:11:59,020 - ERROR - Could not save GraphML file: All strings must be XML compatible: Unicode or ASCII, no NULL bytes or control characters
